In [ ]:
import os
import time
import requests
import pandas as pd
from io import StringIO

In [ ]:
# Create a directory to save the CSV files
output_dir = "Billboard_Top_100_CSVs"
os.makedirs(output_dir, exist_ok=True)

def billboard_top_100_extraction(start_year, end_year):
    # Add headers to mimic a real web browser
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }   
    # Loop through the requested years
    for year in range(2025, 2026):
        print(f"Fetching data for {year}...")
        url = f"https://en.wikipedia.org/wiki/Billboard_Year-End_Hot_100_singles_of_{year}"
    
        try:
            # 1. Fetch the HTML using requests with the browser headers
            response = requests.get(url, headers=headers)
            response.raise_for_status() # Check if the request was successful
        
            # 2. Pass the HTML string to pandas using StringIO
            html_data = StringIO(response.text)
            tables = pd.read_html(html_data)
        
            # Find the table that contains the Billboard Hot 100 list
            df = None
            for table in tables:
                # Normalize column names to title case
                table.columns = [str(col).title() for col in table.columns]
            
                if 'Title' in table.columns and 'Artist(S)' in table.columns:
                    df = table[['Title', 'Artist(S)']]
                    break
                elif 'Title' in table.columns and 'Artist' in table.columns:
                    df = table[['Title', 'Artist']]
                    break
                elif 'Song' in table.columns and 'Artist(S)' in table.columns:
                    df = table[['Song', 'Artist(S)']]
                    break
                elif 'Song' in table.columns and 'Artist' in table.columns:
                    df = table[['Song', 'Artist']]
                    break
            
            if df is not None:
                # Rename columns to match your request
                df.columns = ['Song Name', 'Artist Name']
            
                # Clean up the text
                df['Song Name'] = df['Song Name'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
                df['Song Name'] = df['Song Name'].str.replace('"', '') 
                df['Artist Name'] = df['Artist Name'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
            
                # Save to CSV
                filename = os.path.join(output_dir, f"Billboard_Top_100_{year}.csv")
                df.to_csv(filename, index=False, encoding='utf-8')
                print(f"Successfully saved {filename}")
            else:
                print(f"Could not find the correct table for {year}.")
            
        except Exception as e:
            print(f"Error fetching data for {year}: {e}")
        
        # Sleep for 2 seconds to be polite to Wikipedia's servers
        time.sleep(2)

    print("\nAll done! Check the folder:", output_dir)

In [ ]:
billboard_top_100_extraction(1990,2026)